# Dataset 01 — Material Comparison Analysis

## Objective

Quantitatively analyse the validated batch-experiment results for four ion-exchange materials evaluated for treatment of low-level radioactive liquid waste (LLW).

## Analysis questions

1. Which ion-exchange material produces the highest distribution coefficient (Kd) for each radionuclide?
2. How does radionuclide-specific performance differ between materials?
3. What fraction of activity remains after treatment?
4. Does the material with the highest batch adsorption performance necessarily represent the best engineering choice?

## Data source

The notebook uses the version-controlled Dataset 01 CSV:

`data/material_comparison/dataset_01_material_comparison.csv`

The dataset contains source experimental values together with the derived `remaining_activity_pct` field documented in the accompanying data dictionary.

## 1. Environment and imports

The analysis uses standard Python data-analysis libraries. No statistical inference or machine-learning model is introduced at this stage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')

## 2. Load Dataset 01

The relative path below assumes that the notebook is executed from the repository root. If the notebook is opened in another working directory, adjust the path without modifying the underlying dataset.

In [ ]:
DATA_PATH = '../../data/material_comparison/dataset_01_material_comparison.csv'

df = pd.read_csv(DATA_PATH)
df.head()

## 3. Basic dataset inspection

In [ ]:
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')

df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
print('Materials:', df['material'].unique().tolist())
print('Analytes:', df['analyte'].unique().tolist())
print('Equilibration times:', df['equilibration_time_h'].unique().tolist())

## 4. Data-quality checks

Dataset 01 is expected to contain 16 observations: four materials × four analytes.

In [ ]:
expected_rows = 4 * 4
assert len(df) == expected_rows, f'Expected {expected_rows} rows, found {len(df)}'

expected_materials = {'4A', '13X-CFC', 'HMO-PU', 'CFC-PU'}
expected_analytes = {'Gross beta-gamma', '90Sr', '137Cs', '99Tc'}

assert set(df['material']) == expected_materials
assert set(df['analyte']) == expected_analytes
assert df[['initial_activity_bq_ml', 'final_activity_bq_ml', 'remaining_activity_pct', 'kd_ml_g']].notna().all().all()

duplicates = df.duplicated(subset=['material', 'analyte']).sum()
assert duplicates == 0, f'Found {duplicates} duplicate material/analyte combinations'

print('All Dataset 01 structural checks passed.')

In [ ]:
# Verify the derived remaining-activity field.
calculated_remaining = (
    df['final_activity_bq_ml'] / df['initial_activity_bq_ml'] * 100
)

max_difference = (calculated_remaining - df['remaining_activity_pct']).abs().max()
print(f'Maximum difference in remaining_activity_pct: {max_difference:.10g}')

assert np.allclose(
    calculated_remaining,
    df['remaining_activity_pct'],
    rtol=1e-6,
    atol=1e-6
)

print('Derived-field validation passed.')

## 5. Validation status

At this stage the notebook performs only structural and data-quality validation. Analytical rankings, visualizations and engineering interpretation will be added in subsequent sections after the baseline dataset passes these checks.